# LLM Request Triage: BurstGPT End-to-End Validation

This notebook installs the shared project package, runs the full test suite, prepares the pinned BurstGPT trace, executes FCFS, length-aware, and static SLO-aware policies through the canonical simulator and evaluation engine, validates request accounting, and visualizes the resulting metrics. A standard CPU Colab runtime is sufficient because inference is simulated.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/xfgao210/llm-request-triage.git"
BRANCH = "src-layout-integration"  # Change to main after the integration PR merges.

is_colab = Path("/content").exists()
if is_colab:
    repo = Path("/content/llm-request-triage")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "switch", BRANCH], check=True)
        subprocess.run(
            ["git", "-C", str(repo), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
else:
    repo = Path.cwd()
    if repo.name == "notebooks":
        repo = repo.parent

os.chdir(repo)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-e", ".[dev]",
        "pandas>=2.0", "matplotlib>=3.7",
    ],
    check=True,
)
print(f"Repository ready at {repo} on branch {BRANCH if is_colab else 'current local branch'}")

## Run the full test suite

Pytest is required because it includes simulator function tests and pytest subtests that unittest discovery does not collect.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    check=True,
)

## Prepare the pinned BurstGPT trace

This downloads the approximately 51 MB pinned trace and license, verifies their hashes, and creates deterministic train, validation, and test windows under the ignored data directory. A verified cache is reused on subsequent runs.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/prepare_burstgpt.py"],
    check=True,
)

## Run the full BurstGPT E2E workflow

The default configuration evaluates eight validation windows at three target request rates with three deterministic policies. Outputs go to a disposable directory rather than overwriting retained repository evidence.

In [ ]:
output_root = (
    Path("/content/llm-triage-e2e")
    if is_colab
    else Path(tempfile.gettempdir()) / "llm-triage-e2e"
)
runs_dir = output_root / "runs"
summary_dir = output_root / "summary"

subprocess.run(
    [
        sys.executable,
        "experiments/run_burstgpt.py",
        "--processed-dir", "data/processed",
        "--output", str(runs_dir),
        "--summary-dir", str(summary_dir),
    ],
    check=True,
)
print(f"E2E artifacts written to {output_root}")

## Validate and inspect the results

A successful default run produces 72 policy runs and 18,432 request executions. Every arrival must end as completed, dropped, or unfinished.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.read_csv(summary_dir / "summary.csv")

assert len(summary) == 72, f"Expected 72 policy runs, found {len(summary)}"
assert summary["arrivals"].sum() == 18_432
assert set(summary["policy"]) == {"fcfs", "length_aware", "static_slo_aware"}
assert set(summary["window"]) == set(range(8))
assert set(summary["target_rps"]) == {0.6, 1.2, 2.4}
assert (
    summary["completed"] + summary["dropped"] + summary["unfinished"]
    == summary["arrivals"]
).all()

display(summary[[
    "policy", "window", "target_rps", "arrivals", "completed",
    "dropped", "unfinished", "slo_attainment", "p95_ttft_s",
    "window_goodput_rps",
]])

policy_summary = summary.groupby("policy").agg(
    mean_slo_attainment=("slo_attainment", "mean"),
    mean_drop_rate=("drop_rate", "mean"),
    mean_p95_ttft_s=("p95_ttft_s", "mean"),
    mean_goodput_rps=("window_goodput_rps", "mean"),
)
display(policy_summary)

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
policy_summary.plot.bar(
    y="mean_slo_attainment", ylim=(0, 1), ax=axes[0], legend=False,
    title="Mean SLO attainment",
)
policy_summary.plot.bar(
    y="mean_goodput_rps", ax=axes[1], legend=False,
    title="Mean window goodput",
)
for axis in axes:
    axis.set_xlabel("")
    axis.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

print("E2E validation passed: 72 policy runs and 18,432 request executions.")

## Optional: artificial KV-capacity stress scenario

Set RUN_KV_STRESS = True to exercise constrained batching, KV allocation, preemption, proactive drops, and unfinished-work accounting. This scenario is supplemental and does not represent physical RTX 5070 capacity.

In [ ]:
RUN_KV_STRESS = False

if RUN_KV_STRESS:
    stress_runs_dir = output_root / "kv-stress-runs"
    stress_summary_dir = output_root / "kv-stress-summary"
    subprocess.run(
        [
            sys.executable,
            "experiments/run_burstgpt.py",
            "--config", "configs/burstgpt_kv_stress.json",
            "--processed-dir", "data/processed",
            "--output", str(stress_runs_dir),
            "--summary-dir", str(stress_summary_dir),
        ],
        check=True,
    )
    stress = pd.read_csv(stress_summary_dir / "summary.csv")
    assert len(stress) == 24
    assert stress["arrivals"].sum() == 6_144
    assert (
        stress["completed"] + stress["dropped"] + stress["unfinished"]
        == stress["arrivals"]
    ).all()
    display(stress[[
        "policy", "window", "completed", "dropped", "unfinished",
        "slo_attainment", "preemptions", "p95_ttft_s",
    ]])
    print("KV-stress validation passed: 24 policy runs and 6,144 executions.")
else:
    print("KV-stress run skipped. Set RUN_KV_STRESS = True to enable it.")

## What this validates

The workflow uses the same BurstGPT preparation, canonical TriageKernel, scheduler callbacks, lifecycle events, metrics collector, and reporting code used by repository experiments. It validates the integrated simulation pipeline; it does not execute real LLM inference or claim measured GPU performance.